In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "archive/src/experiment.py").is_file()
)
PROJECT_ROOT = ROOT = REPOSITORY_ROOT / "archive"
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))


# Две финальные AGMFA HardShift — фиксированные 22 эпохи
Свежая ImageNet V2 initialization для обеих моделей. Corrected AGMFA conv2 taps,
partial layer3/4, frozen BN, warmup3, прежние AdamW/LR/seed42, cosine T_max22.
Batch16 **объектов**: source → clean+hard (2 views), бывший val → только D4 (1 view).

**Train-only:** val только для отчёта; last.pt эпохи22 — финальный.
**Train+val:** бывший val получает только D4 и CE weight2, без synthetic corruptions.
Обычное shuffled объединение, каждый объект один раз за эпоху, никакого oversampling.
Loss = (sum(source losses)+2 sum(target CE))/(n_source+2 n_target).
Validation selection отсутствует; final.pt эпохи22. Старые checkpoints не используются.
22 эпохи заданы заранее; local peak эпохи14 не участвует в выборе.



In [ ]:
from pathlib import Path
import sys,json
import pandas as pd
import torch
from IPython.display import display

ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':
    ROOT=ROOT.parent
if not (ROOT/'src').is_dir():
    raise RuntimeError('Укажите ROOT — корень проекта с src/configs')
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from archive.src.experiment import load_config
from archive.src.final_hardshift import run_final_hardshift,plot_final_history,validate_final_config
from archive.src.utils import get_device
from archive.src.hard_target_shift import PROFILES


In [ ]:
RUN_TRAIN_ONLY=False
RUN_TRAIN_VAL=False

RAW_DIR=REPOSITORY_ROOT / "data/raw"  # Kaggle: каталог с train/val и CSV
DEVICE='auto'
BATCH_SIZE=16  # original objects, source batches ->32 views; reduce to8 if OOM
NUM_WORKERS=2
CPU_THREADS=4
if get_device(DEVICE).type=='cpu':
    torch.set_num_threads(CPU_THREADS)
workers=NUM_WORKERS if get_device(DEVICE).type=='cuda' else 0
TRAIN_CONFIG=ROOT/'configs/resnet50_agmfa_hardshift_final_train.json'
TRAIN_VAL_CONFIG=ROOT/'configs/resnet50_agmfa_hardshift_final_train_val.json'
cfg_train=load_config(TRAIN_CONFIG)
cfg_refit=load_config(TRAIN_VAL_CONFIG)
for cfg in (cfg_train,cfg_refit):
    cfg['data'].update(raw_dir=str(RAW_DIR),batch_size=BATCH_SIZE,num_workers=workers)
    cfg['training']['device']=DEVICE
    validate_final_config(cfg)
print('Device:',get_device(DEVICE),'| fixed epochs:',22)
print('Train-only:',cfg_train['exp_id'])
print('Train+val:',cfg_refit['exp_id'])



In [ ]:
if RUN_TRAIN_ONLY:
    train_result=run_final_hardshift(cfg_train,project_root=ROOT,show_plots=True)
    print(train_result['summary'])
    del train_result  # освобождаем модель перед вторым обучением
else:
    print('Train-only обучение выключено.')


In [ ]:
if RUN_TRAIN_VAL:
    refit_result=run_final_hardshift(cfg_refit,project_root=ROOT,show_plots=True)
    print(refit_result['summary'])
    del refit_result
else:
    print('Train+val обучение выключено.')


## Сохранённые curves и аудит coverage
История каждого run: source clean/hard CE и Accuracy, JS, общий loss, LR.
Train+val дополнительно target CE/Accuracy; train-only — отчётная val loss/Accuracy.
Никакие val показатели не выбирают checkpoint и не управляют scheduler.
Profile counts касаются **только source**. Проверка once-per-epoch для каждого
source и target выполняется внутри training loop. В 22 эпохах три полных
6-эпоховых цикла и четыре эпохи следующего цикла.


In [ ]:
for cfg in (cfg_train,cfg_refit):
    logs=ROOT/'artifacts/logs'/cfg['exp_id']
    if not (logs/'summary.json').is_file():
        print(cfg['exp_id'],': ещё не завершён')
        continue
    summary=json.loads((logs/'summary.json').read_text())
    history=pd.read_csv(logs/'history.csv')
    counts=pd.read_csv(logs/'profile_counts.csv')
    metadata=json.loads((logs/'metadata.json').read_text())
    assert summary['epochs']==22 and summary['selection']=='fixed_last_epoch'
    assert history.epoch.tolist()==list(range(1,23))
    assert metadata['target_transforms']=='D4 only; no hard profile'
    assert not metadata['test_read']
    for epoch in range(1,23):
        expected=pd.Series([(i+epoch)%6 for i in range(4999)]).value_counts().reindex(range(6),fill_value=0)
        actual=counts[counts.epoch==epoch].set_index('profile').reindex(PROFILES)['count'].to_numpy()
        assert (actual==expected.to_numpy()).all()
    if cfg['final_hardshift']['training_splits']==['train','val']:
        assert history.source_samples.eq(4999).all() and history.target_samples.eq(699).all()
        assert not any(name.startswith('val_') for name in history.columns)
    else:
        assert history.target_samples.eq(0).all()
    display(pd.DataFrame([{'exp_id':cfg['exp_id'],**summary}]))
    plot_final_history(history,show=True)
    display(counts.pivot(index='epoch',columns='profile',values='count'))
